# 🤖 Notebook: Agents — *Part 1 of 4*

*Chapter overview: **1) Agents** → 2) MCP → 3) Context Compression & Memory → 4) Subagents*

In this notebook we build agents step by step:

1. **Tool calling with Ollama:** let the LLM call our Python functions
2. **Agents with LangChain:** the same idea, with much less code
3. **ReAct:** the reasoning pattern behind (almost) every agent

## 📚 Sources

- [Ollama: Tool Calling](https://docs.ollama.com/capabilities/tool-calling)
- [LangChain: Agents](https://docs.langchain.com/oss/python/langchain/agents)
- Yao et al. (2022), [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/pdf/2210.03629)

# Part 1: Tool Calling with Ollama

## 1. What is Tool Calling?

An LLM only generates text. It can't look up the current price of a product or do exact math on its own. With **tool calling**, we tell the model which functions ("tools") exist. Instead of answering directly, the model can then reply with a request like *"call `get_price` with `product="laptop"`"*. That request is also just text, but in a fixed format.

**Our code** runs the function and sends the result back. Then the model writes the final answer. The model itself never executes anything.

Not every model was trained for tool calling. `gemma4:26b` was, so we use it here via the `ollama` Python package.

In [ ]:
import os
from dotenv import load_dotenv
from ollama import Client

load_dotenv()  # reads LLM_HOST, LLM_PORT and LLM_API_KEY from the .env file (see setup.md)

LLM_HOST = os.environ["LLM_HOST"]
LLM_PORT = os.environ["LLM_PORT"]
LLM_API_KEY = os.environ["LLM_API_KEY"]
LLM_URL = f"http://{LLM_HOST}:{LLM_PORT}"
LLM_HEADERS = {"Authorization": f"Bearer {LLM_API_KEY}"}
LLM_REASONING = "gemma4:26b"

client = Client(host=LLM_URL, headers=LLM_HEADERS)

## 2. A First Tool

Our example is a small online shop. A tool is a normal Python function. The `ollama` package builds the tool description for the model from the **type hints** and the **docstring**, so both matter: they are all the model knows about the function.

In [ ]:
def get_price(product: str) -> str:
    """Get the current price of a product in our shop, in euros.

    Args:
        product: The product name, e.g. laptop, headphones or mouse
    """
    prices = {"laptop": "999", "headphones": "149", "mouse": "25"}
    return prices.get(product.lower(), "Unknown product")

In [ ]:
messages = [{"role": "user", "content": "How much does the laptop cost?"}]

response = client.chat(model=LLM_REASONING, messages=messages, tools=[get_price])

print("Content:", response.message.content)
print("Tool calls:", response.message.tool_calls)

The model did not answer. It asked for a tool call instead. Now we:

1. add the model's message to the conversation,
2. run the function ourselves,
3. add the result as a message with `role: "tool"`,
4. ask the model again.

In [ ]:
messages.append(response.message)  # 1.

call = response.message.tool_calls[0]
result = get_price(**call.function.arguments)  # 2.
messages.append({"role": "tool", "tool_name": call.function.name, "content": result})  # 3.

final = client.chat(model=LLM_REASONING, messages=messages, tools=[get_price])  # 4.
print(final.message.content)

If no tool is needed, the model just answers and `tool_calls` is `None`:

In [ ]:
response = client.chat(
    model=LLM_REASONING,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
    tools=[get_price],
)
print("Tool calls:", response.message.tool_calls)
print("Content:", response.message.content)

### Exercise 1: A Tool with a Real API

Write a tool `get_exchange_rate(currency: str) -> str` that fetches the **live** exchange rate from euros to another currency from [Frankfurter](https://frankfurter.dev) (free, no API key). Then ask the model: *"How many US dollars do I get for one euro today?"*, using the same four steps as above.

This is how the API works:

In [ ]:
import requests

data = requests.get("https://api.frankfurter.dev/v1/latest?from=EUR&to=USD").json()
print(data)
print(data["rates"]["USD"])

<details>
<summary><b>Show Solution</b></summary>

```python
def get_exchange_rate(currency: str) -> str:
    """Get the current exchange rate from 1 euro to another currency.

    Args:
        currency: The three-letter currency code, e.g. USD, GBP or JPY
    """
    data = requests.get(f"https://api.frankfurter.dev/v1/latest?from=EUR&to={currency}").json()
    return str(data["rates"][currency])

messages = [{"role": "user", "content": "How many US dollars do I get for one euro today?"}]
response = client.chat(model=LLM_REASONING, messages=messages, tools=[get_exchange_rate])
messages.append(response.message)

call = response.message.tool_calls[0]
messages.append({"role": "tool", "tool_name": call.function.name, "content": get_exchange_rate(**call.function.arguments)})

final = client.chat(model=LLM_REASONING, messages=messages, tools=[get_exchange_rate])
print(final.message.content)
```

</details>

In [ ]:
# Your code goes here...

## 3. Several Tools

With more than one tool, we need to find the right Python function for each call. A dict from name to function does that. The model may also request **several calls at once**, so we loop over all of them.

In [ ]:
def get_stock(product: str) -> str:
    """Get how many units of a product are currently in stock.

    Args:
        product: The product name, e.g. laptop, headphones or mouse
    """
    stock = {"laptop": "3", "headphones": "0", "mouse": "42"}
    return stock.get(product.lower(), "Unknown product")

available_functions = {
    "get_price": get_price,
    "get_stock": get_stock,
}

In [ ]:
messages = [{"role": "user", "content": "What do the laptop and the headphones cost, and are they in stock?"}]
tools = list(available_functions.values())

response = client.chat(model=LLM_REASONING, messages=messages, tools=tools)
messages.append(response.message)

for call in response.message.tool_calls:
    result = available_functions[call.function.name](**call.function.arguments)
    print(f"{call.function.name}({call.function.arguments}) -> {result}")
    messages.append({"role": "tool", "tool_name": call.function.name, "content": result})

final = client.chat(model=LLM_REASONING, messages=messages, tools=tools)
print(final.message.content)

## 4. The Agent Loop

Sometimes the model needs the result of one tool before it knows the next step. So we repeat in a loop: call the model → run the requested tools → send the results back. We stop as soon as the model answers without a tool call.

**This loop is an agent.** The LLM decides step by step what to do, our code carries it out.

In [ ]:
def add(a: int, b: int) -> int:
    """Add two numbers.

    Args:
        a: The first number
        b: The second number
    """
    return a + b


def multiply(a: int, b: int) -> int:
    """Multiply two numbers.

    Args:
        a: The first number
        b: The second number
    """
    return a * b


math_functions = {"add": add, "multiply": multiply}

In [ ]:
messages = [{"role": "user", "content": "What is (11434 + 12341) * 412?"}]

while True:
    response = client.chat(model=LLM_REASONING, messages=messages, tools=list(math_functions.values()))
    messages.append(response.message)

    if not response.message.tool_calls:  # no more tool calls -> final answer
        break

    for call in response.message.tool_calls:
        result = math_functions[call.function.name](**call.function.arguments)
        print(f"{call.function.name}({call.function.arguments}) -> {result}")
        messages.append({"role": "tool", "tool_name": call.function.name, "content": str(result)})

print(response.message.content)

The model first called `add` and only then `multiply` with the result. It could not have done both at once.

### Exercise 2: Agent Loop with a Discount

Write a tool `apply_discount(price: float, percent: float) -> float` and run the agent loop with `get_price` and your new tool. Ask: *"How much does the laptop cost with a 15% discount?"*

The model needs two steps: first the price, then the discount.

<details>
<summary><b>Show Solution</b></summary>

```python
def apply_discount(price: float, percent: float) -> float:
    """Apply a percentage discount to a price and return the new price.

    Args:
        price: The original price
        percent: The discount in percent, e.g. 15 for 15%
    """
    return round(price * (1 - percent / 100), 2)

functions = {"get_price": get_price, "apply_discount": apply_discount}
messages = [{"role": "user", "content": "How much does the laptop cost with a 15% discount?"}]

while True:
    response = client.chat(model=LLM_REASONING, messages=messages, tools=list(functions.values()))
    messages.append(response.message)
    if not response.message.tool_calls:
        break
    for call in response.message.tool_calls:
        result = functions[call.function.name](**call.function.arguments)
        print(f"{call.function.name}({call.function.arguments}) -> {result}")
        messages.append({"role": "tool", "tool_name": call.function.name, "content": str(result)})

print(response.message.content)
```

</details>

In [ ]:
# Your code goes here...

# Part 2: Agents with LangChain

Writing the loop by hand is great for understanding, but tedious. **[LangChain](https://www.langchain.com/)** is a framework that provides building blocks for LLM apps (models, tools, agents) that work the same way for every provider. `create_agent` contains exactly the loop from above.

Three building blocks:

- `ChatOllama`: the model, connected to our Ollama server
- `@tool`: turns a Python function into a tool (again via type hints + docstring)
- `create_agent`: builds the agent loop around model and tools

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.tools import tool

llm = ChatOllama(model=LLM_REASONING, base_url=LLM_URL, client_kwargs={"headers": LLM_HEADERS}, temperature=0)


@tool
def get_price(product: str) -> str:
    """Get the current price of a product in our shop, in euros."""
    prices = {"laptop": "999", "headphones": "149", "mouse": "25"}
    return prices.get(product.lower(), "Unknown product")


agent = create_agent(model=llm, tools=[get_price])

result = agent.invoke({"messages": [{"role": "user", "content": "How much does the laptop cost?"}]})
for message in result["messages"]:
    print(f"{type(message).__name__}: {message.content!r} {getattr(message, 'tool_calls', '') or ''}")

`result["messages"]` is the full conversation, exactly the same steps as in Part 1: `HumanMessage` → `AIMessage` with a tool call → `ToolMessage` with the result → `AIMessage` with the final answer.

With `system_prompt` you give the agent general instructions:

In [ ]:
pirate_agent = create_agent(
    model=llm,
    tools=[get_price],
    system_prompt="You are a shop assistant. Always answer in one short sentence, like a pirate.",
)

result = pirate_agent.invoke({"messages": [{"role": "user", "content": "How much are the headphones?"}]})
print(result["messages"][-1].content)

<details>
<summary><b>What does this look like raw?</b></summary>

In the end, every library (`ollama`, `openai`, `langchain`) produces plain text for the model. Rendered with Gemma 4's chat template, the exchange above looks roughly like this:

```md
<bos><|turn>system
<|tool>declaration:get_price{description:<|"|>Get the current price of a product in our shop, in euros.<|"|>,parameters:{properties:{product:{type:<|"|>STRING<|"|>}},required:[<|"|>product<|"|>],type:<|"|>OBJECT<|"|>}}<tool|><turn|>
<|turn>user
How much does the laptop cost?<turn|>
<|turn>model
<|tool_call>call:get_price{product:<|"|>laptop<|"|>}<tool_call|><|tool_response>response:get_price{value:<|"|>999<|"|>}<tool_response|>The laptop costs 999 euros.<turn|>
```

- The tool descriptions are inserted once at the start, in a system turn.
- The tool call and the tool response are special tokens inside the model's turn.
- `AIMessage.tool_calls` and `ToolMessage` are just a convenient Python view of this text.

</details>

### Exercise 3: Exercise 2 with LangChain

Build Exercise 2 (laptop with a 15% discount) again, this time with `@tool` and `create_agent`. Compare how much code you need.

<details>
<summary><b>Show Solution</b></summary>

```python
@tool
def apply_discount(price: float, percent: float) -> float:
    """Apply a percentage discount (e.g. 15 for 15%) to a price and return the new price."""
    return round(price * (1 - percent / 100), 2)

agent = create_agent(model=llm, tools=[get_price, apply_discount])
result = agent.invoke({"messages": [{"role": "user", "content": "How much does the laptop cost with a 15% discount?"}]})
print(result["messages"][-1].content)
```

</details>

In [ ]:
# Your code goes here...

# Part 3: ReAct

**ReAct** ([Yao et al., 2022](https://arxiv.org/pdf/2210.03629)) stands for **Re**asoning + **Act**ing. The agent repeats three steps until it can answer:

- **Thought:** think about what to do next
- **Action:** call a tool
- **Observation:** look at the tool result

That is exactly our agent loop. With a reasoning model like `gemma4:26b`, the *Thought* is the model's thinking. Let's make it visible with `reasoning=True` and a question that needs two steps one after the other.

In [ ]:
@tool
def get_author(book: str) -> str:
    """Get the author of a book."""
    authors = {"Deep Learning": "Ian Goodfellow", "Introduction to Algorithms": "Thomas H. Cormen"}
    return authors.get(book, "Unknown book")


@tool
def get_birth_year(person: str) -> str:
    """Get the birth year of a person."""
    years = {"Ian Goodfellow": "1985", "Thomas H. Cormen": "1956"}
    return years.get(person, "Unknown person")


llm_thinking = ChatOllama(model=LLM_REASONING, base_url=LLM_URL, client_kwargs={"headers": LLM_HEADERS}, temperature=0, reasoning=True)
react_agent = create_agent(model=llm_thinking, tools=[get_author, get_birth_year])

result = react_agent.invoke({"messages": [{"role": "user", "content": "In which year was the author of 'Deep Learning' born?"}]})

In [ ]:
def print_react_trace(messages):
    for m in messages:
        if m.type == "human":
            print("QUESTION:   ", m.content)
        elif m.type == "ai":
            if m.additional_kwargs.get("reasoning_content"):
                print("THOUGHT:    ", m.additional_kwargs["reasoning_content"].strip())
            for call in m.tool_calls:
                print("ACTION:     ", f"{call['name']}({call['args']})")
            if m.content:
                print("ANSWER:     ", m.content)
        elif m.type == "tool":
            print("OBSERVATION:", m.content)
        print()

print_react_trace(result["messages"])

The agent first finds the author and only then asks for the birth year. Each thought builds on the previous observation.

### Exercise 4: Your Own Shopping Agent

Build an agent with three tools and print its ReAct trace with `print_react_trace`:

- `get_price(product)`: from Part 2
- `apply_discount(price, percent)`: from Exercise 3
- `get_exchange_rate(currency)`: the live exchange rate from Exercise 1

Ask: *"What does the laptop cost in US dollars with a 10% discount?"*

<details>
<summary><b>Show Solution</b></summary>

```python
@tool
def apply_discount(price: float, percent: float) -> float:
    """Apply a percentage discount (e.g. 15 for 15%) to a price and return the new price."""
    return round(price * (1 - percent / 100), 2)

@tool
def get_exchange_rate(currency: str) -> str:
    """Get the current exchange rate from 1 euro to another currency (three-letter code, e.g. USD)."""
    data = requests.get(f"https://api.frankfurter.dev/v1/latest?from=EUR&to={currency}").json()
    return str(data["rates"][currency])

shop_agent = create_agent(model=llm_thinking, tools=[get_price, apply_discount, get_exchange_rate])
result = shop_agent.invoke({"messages": [{"role": "user", "content": "What does the laptop cost in US dollars with a 10% discount?"}]})
print_react_trace(result["messages"])
```

</details>

In [ ]:
# Your code goes here...